# RLM pre-screening experiment: analysis

Analyses an experiment written by `examples/run-rlm-screening.py`: plain LLaMEA
(`baseline-no-screening`) against LLaMEA with RLM pre-screening
(`log-only`, `topk-k50`, `topk-k25`, `threshold-q50`, per checkpoint).

1. **Status**: which runs are finished, failed or still pending.
2. **Final performance**: best fitness (AOCC) per method at the same number of
   real evaluations, with non-parametric tests against the baseline.
3. **Convergence**: best-so-far fitness against real evaluations, and the
   area under that curve (anytime performance).
4. **Cost**: LLM calls and wall-clock time per run.
5. **RLM prediction quality**: on the `log-only` arm, which scores every
   offspring but never filters, so its predictions are an unbiased sample.
6. **Screened arms**: offspring quality and failure rate of the candidates the
   screening arms did evaluate.

Only runs recorded as finished in `experimentlog.jsonl` are analysed.

In [ ]:
import json
import os
import re

import matplotlib.patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from iohblade.plots import fitness_table, plot_boxplot_fitness, plot_convergence
from iohblade.loggers import ExperimentLogger

# Path to the experiment, relative to this notebook (examples/).
RESULTS_DIR = os.environ.get("RLM_RESULTS_DIR", "../results/rlm-screening-mabbob")
PROBLEM = "MA_BBOB"
BASELINE = "baseline-no-screening"
BUDGET = 100  # real evaluations per run
SAVE_FIGS = False  # True: also write PNGs to RESULTS_DIR/figures/

logger = ExperimentLogger(RESULTS_DIR, read=True)
FIG_DIR = os.path.join(RESULTS_DIR, "figures")


def save(fig, name):
    if SAVE_FIGS:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=200, bbox_inches="tight")

### Method colours and styles
Colour encodes the **screening strategy**; line style and hatching encode the
**RLM checkpoint** (solid = lhs200, dashed = lhs20), so the same strategy looks
alike across checkpoints. The categorical palette is colour-blind checked; every
figure also has a legend or a table.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
STRATEGY_ORDER = ["baseline-no-screening", "log-only", "topk-k50", "topk-k25", "threshold-q50"]
CHECKPOINT_STYLE = {"none": "-", "lhs200": "-", "lhs20": "--"}
CHECKPOINT_HATCH = {"none": "", "lhs200": "", "lhs20": "//"}


def strategy_of(method):
    return re.sub(r"-lhs\d+", "", method)


def checkpoint_of(method):
    m = re.search(r"lhs\d+", method)
    return m.group(0) if m else "none"


def order_methods(methods):
    known = {s: i for i, s in enumerate(STRATEGY_ORDER)}
    return sorted(
        methods,
        key=lambda m: (known.get(strategy_of(m), len(known)), strategy_of(m), checkpoint_of(m) != "lhs200", m),
    )


exp = logger.get_data()
if exp.empty:
    raise SystemExit(f"No finished runs in {RESULTS_DIR}/experimentlog.jsonl")
METHODS = order_methods(exp["method_name"].unique())
strategies = list(dict.fromkeys(strategy_of(m) for m in METHODS))
STRATEGY_COLOR = {s: PALETTE[i % len(PALETTE)] for i, s in enumerate(strategies)}
COLOR = {m: STRATEGY_COLOR[strategy_of(m)] for m in METHODS}
STYLE = {m: CHECKPOINT_STYLE.get(checkpoint_of(m), ":") for m in METHODS}
HATCH = {m: CHECKPOINT_HATCH.get(checkpoint_of(m), "..") for m in METHODS}
SCREENED = [m for m in METHODS if m != BASELINE]
LOG_ONLY = [m for m in METHODS if strategy_of(m) == "log-only"]


def hatch_boxes(ax, order):
    # seaborn's patch order isn't the category order once hue is set, so match
    # each box to its category by its x position (category i is centred at x=i).
    for patch in ax.patches:
        if not isinstance(patch, matplotlib.patches.PathPatch):
            continue
        i = int(round(patch.get_path().vertices[:, 0].mean()))
        if 0 <= i < len(order):
            patch.set_hatch(HATCH[order[i]])

sns.set_theme(style="whitegrid", context="notebook")
print("Methods:", METHODS)

## 1. Status of the runs

In [ ]:
with open(os.path.join(RESULTS_DIR, "progress.json")) as f:
    progress = json.load(f)

status = pd.DataFrame(progress["runs"])
status["state"] = np.select(
    [status["end_time"].notna(), status.get("error", pd.Series(index=status.index)).notna()],
    ["finished", "failed"],
    default="pending",
)
status_table = (
    status.pivot_table(index="method_name", columns="state", values="seed", aggfunc="count", fill_value=0)
    .reindex(order_methods(status["method_name"].unique()))
)
display(status_table)
failed = status[status["state"] == "failed"]
if len(failed):
    print("Failed runs (traceback in error.txt in each run folder):")
    display(failed[["method_name", "seed", "error"]])

## Load the candidates

One row per **real evaluation** (`log.jsonl`), loaded with BLADE's
`ExperimentLogger.get_problem_data`. Candidates that `topk_filter` discarded or
`threshold_reject` rejected were never evaluated, so they are not here.
Invalid candidates (code that crashed) have fitness `-inf`; where an average
needs a number they count as 0, the lowest AOCC.

In [ ]:
cand = logger.get_problem_data(problem_name=PROBLEM).drop(columns=["code"], errors="ignore")
cand["fitness"] = cand["fitness_scalar"].astype(float)
cand["valid"] = np.isfinite(cand["fitness"])
cand["fit0"] = cand["fitness"].where(cand["valid"], 0.0)  # invalid -> 0 (lowest AOCC)
cand["eval"] = cand["_id"] + 1
cand["rlm_score"] = cand["metadata"].apply(
    lambda md: md.get("rlm_predicted_score", np.nan) if isinstance(md, dict) else np.nan
)
cand = cand[cand["method_name"].isin(METHODS)]


def llm_calls(log_dir):
    path = os.path.join(RESULTS_DIR, log_dir, "conversationlog.jsonl")
    if not os.path.exists(path):
        return np.nan
    with open(path) as f:
        return sum(json.loads(line).get("role") == "client" for line in f if line.strip())


def best_so_far(values, budget=BUDGET):
    curve = np.maximum.accumulate(np.asarray(values, dtype=float))
    if len(curve) < budget:  # a run that ended early keeps its last value
        curve = np.concatenate([curve, np.full(budget - len(curve), curve[-1] if len(curve) else 0.0)])
    return curve[:budget]


times = status.set_index(["method_name", "seed"])[["start_time", "end_time"]]
rows = []
for _, run in exp.iterrows():
    c = cand[(cand["method_name"] == run["method_name"]) & (cand["seed"] == run["seed"])].sort_values("eval")
    curve = best_so_far(c["fit0"])
    start, end = times.loc[(run["method_name"], run["seed"])] if (run["method_name"], run["seed"]) in times.index else (None, None)
    rows.append(
        {
            "method_name": run["method_name"],
            "seed": run["seed"],
            "best": c["fit0"].max(),
            "anytime": curve.mean(),  # area under best-so-far / BUDGET
            "evals": len(c),
            "invalid_frac": 1 - c["valid"].mean(),
            "generations": c["generation"].nunique(),
            "llm_calls": llm_calls(run["log_dir"]),
            "hours": (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds() / 3600 if start and end else np.nan,
        }
    )
runs = pd.DataFrame(rows)
print(f"{len(runs)} finished runs, {len(cand)} evaluated candidates")
runs.groupby("method_name").size().reindex(METHODS)

## 2. Final performance

Best fitness (AOCC, higher is better) after the full budget of real
evaluations. Each method is compared with the baseline using a two-sided
Mann-Whitney U test (Holm-corrected over methods) and the Vargha-Delaney
effect size $\hat A_{12}$ = P(method > baseline): 0.5 is no difference, and
0.56 / 0.64 / 0.71 are the usual small / medium / large thresholds. With 5 runs
per method only large effects can reach significance.

In [ ]:
def a12(x, y):
    # Vargha-Delaney A12: P(X > Y) + 0.5 P(X = Y)
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = (x[:, None] > y[None, :]).mean()
    eq = (x[:, None] == y[None, :]).mean()
    return gt + 0.5 * eq


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(np.where(np.isnan(p), np.inf, p))
    adj, running = np.full_like(p, np.nan), 0.0
    m = np.sum(~np.isnan(p))
    for rank, i in enumerate(order):
        if np.isnan(p[i]):
            continue
        running = max(running, min(1.0, (m - rank) * p[i]))
        adj[i] = running
    return adj


def compare_to_baseline(metric):
    base = runs.loc[runs["method_name"] == BASELINE, metric].dropna()
    out = []
    for m in METHODS:
        x = runs.loc[runs["method_name"] == m, metric].dropna()
        p = np.nan
        if m != BASELINE and len(x) >= 2 and len(base) >= 2:
            p = stats.mannwhitneyu(x, base, alternative="two-sided").pvalue
        out.append(
            {
                "method": m,
                "n": len(x),
                "mean": x.mean(),
                "std": x.std(),
                "median": x.median(),
                "A12 vs baseline": a12(x, base) if m != BASELINE else np.nan,
                "p (MWU)": p,
            }
        )
    df = pd.DataFrame(out).set_index("method")
    df["p (Holm)"] = holm(df["p (MWU)"])
    return df


summary = compare_to_baseline("best")
display(summary.style.format(precision=3, na_rep="–"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=runs, x="method_name", y="best", order=METHODS, palette=COLOR, width=0.55,
            fliersize=0, linewidth=1, ax=ax, hue="method_name", legend=False)
hatch_boxes(ax, METHODS)
sns.stripplot(data=runs, x="method_name", y="best", order=METHODS, color="#0b0b0b", size=6,
              jitter=0.12, ax=ax)
ax.set(xlabel="", ylabel="Best AOCC after the budget", title="Final best fitness per run")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
save(fig, "final_best")

BLADE's own views of the same results: its box plot, and its table of mean ±
std per method. Its p-value is the largest of Welch's t-tests against every
other method, and bold marks a method better than all others.

In [ ]:
plot_boxplot_fitness(logger, y_label="Best AOCC")
fitness_table(logger, alpha=0.05)

## 3. Convergence

Best-so-far fitness against the number of **real** evaluations, the budget
every method shares. The first plot is BLADE's `plot_convergence` (mean ± std).
The second shows the median with the interquartile range, which is more robust
with 5 runs per method. The last panel compares the **anytime performance**:
the area under each run's best-so-far curve divided by the budget.

In [ ]:
plot_convergence(logger, metric="AOCC", budget=BUDGET, save=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(1, BUDGET + 1)
for m in METHODS:
    curves = []
    for seed, c in cand[cand["method_name"] == m].groupby("seed"):
        curves.append(best_so_far(c.sort_values("eval")["fit0"]))
    if not curves:
        continue
    curves = np.vstack(curves)
    med = np.median(curves, axis=0)
    lo, hi = np.percentile(curves, [25, 75], axis=0)
    ax.plot(x, med, color=COLOR[m], linestyle=STYLE[m], linewidth=2, label=f"{m} (n={len(curves)})")
    ax.fill_between(x, lo, hi, color=COLOR[m], alpha=0.12, linewidth=0)
ax.set(xlabel="Real evaluations", ylabel="Best-so-far AOCC (median, IQR)", xlim=(1, BUDGET),
       title="Convergence per method")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
save(fig, "convergence_median")

In [ ]:
anytime = compare_to_baseline("anytime")
display(anytime.style.format(precision=3, na_rep="–"))

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=runs, x="method_name", y="anytime", order=METHODS, palette=COLOR, width=0.55,
            fliersize=0, linewidth=1, ax=ax, hue="method_name", legend=False)
hatch_boxes(ax, METHODS)
sns.stripplot(data=runs, x="method_name", y="anytime", order=METHODS, color="#0b0b0b", size=6, jitter=0.12, ax=ax)
ax.set(xlabel="", ylabel="Area under best-so-far / budget", title="Anytime performance")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
save(fig, "anytime")

## 4. Cost

Screening changes what a run spends besides real evaluations: `topk_filter`
asks the LLM for 2× (k50) or 4× (k25) as many candidates per generation, and
`threshold_reject` runs more generations to reach the same number of real
evaluations. LLM calls are counted from each run's `conversationlog.jsonl`.
(BLADE's token-cost estimates come from `tokencost`, which doesn't know every
Claude model, so they may read 0. Check the Claude Console for actual spend.)

In [ ]:
cost = runs.groupby("method_name")[["llm_calls", "hours", "evals", "invalid_frac", "generations"]].mean().reindex(METHODS)
display(cost.style.format({"llm_calls": "{:.0f}", "hours": "{:.2f}", "evals": "{:.0f}",
                           "invalid_frac": "{:.1%}", "generations": "{:.1f}"}, na_rep="–"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for m in METHODS:
    r = runs[runs["method_name"] == m]
    axes[0].scatter(r["llm_calls"], r["best"], s=60, color=COLOR[m], label=m,
                    marker="o" if checkpoint_of(m) != "lhs20" else "s", edgecolor="white", linewidth=1.5)
    axes[1].scatter(r["hours"], r["best"], s=60, color=COLOR[m], label=m,
                    marker="o" if checkpoint_of(m) != "lhs20" else "s", edgecolor="white", linewidth=1.5)
axes[0].set(xlabel="LLM calls per run", ylabel="Best AOCC", title="Fitness vs LLM calls")
axes[1].set(xlabel="Wall-clock hours per run", ylabel="Best AOCC", title="Fitness vs run time")
axes[1].legend(frameon=False, fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
save(fig, "cost")

## 5. RLM prediction quality (log-only arm)

The `log-only` arm scores every offspring but evaluates all of them, so its
(prediction, real fitness) pairs are an **unbiased** sample of what the
screener sees. The filtering arms only evaluate the candidates the RLM already
preferred, so their pairs are biased (section 6).

Each generation's offspring are scored together, and screening only uses the
ranking *within* that batch. So the most relevant numbers here are the
within-batch rank correlation and the counterfactual "what would top-k have
picked" analysis, not the overall correlation.

In [ ]:
lo = cand[cand["method_name"].isin(LOG_ONLY) & cand["rlm_score"].notna()].copy()
if lo.empty:
    print("No log-only runs with RLM predictions yet.")
else:
    val = lo[lo["valid"]]

    def corr_row(df, label):
        if len(df) < 3 or df["rlm_score"].nunique() < 2 or df["fitness"].nunique() < 2:
            return {"set": label, "n": len(df)}
        return {
            "set": label,
            "n": len(df),
            "Spearman": stats.spearmanr(df["rlm_score"], df["fitness"]).statistic,
            "Kendall": stats.kendalltau(df["rlm_score"], df["fitness"]).statistic,
            "Pearson": stats.pearsonr(df["rlm_score"], df["fitness"]).statistic,
        }

    rows = [corr_row(val[val["method_name"] == m], f"{m}: all valid") for m in LOG_ONLY]
    rows += [corr_row(g, f"{m}: seed {s}") for (m, s), g in val.groupby(["method_name", "seed"])]
    display(pd.DataFrame(rows).set_index("set").style.format(precision=3, na_rep="–"))

    fig, axes = plt.subplots(1, len(LOG_ONLY), figsize=(6 * len(LOG_ONLY), 5), squeeze=False)
    for ax, m in zip(axes[0], LOG_ONLY):
        d = lo[lo["method_name"] == m]
        v, inv = d[d["valid"]], d[~d["valid"]]
        ax.scatter(v["rlm_score"], v["fitness"], s=24, color=COLOR[m], alpha=0.6, edgecolor="white",
                   linewidth=0.5, label="valid")
        if len(inv):
            ax.scatter(inv["rlm_score"], np.zeros(len(inv)), s=24, marker="x", color="#52514e",
                       label="invalid (plotted at 0)")
        ax.set(xlabel="RLM predicted score", ylabel="Real AOCC", title=m)
        ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    save(fig, "rlm_scatter")

**Within-batch ranking.** For every generation batch (one `(run, generation)`)
the Spearman correlation between prediction and real fitness, using invalid
candidates as 0. A distribution centred above 0 means the RLM ranks offspring
better than chance.

**Invalid candidates.** The AUROC for telling valid from invalid candidates by
predicted score: 0.5 is chance, 1 is perfect. Screening that removes crashing
code is useful on its own.

In [ ]:
if not lo.empty:
    batches = lo[lo["generation"] > 0].groupby(["method_name", "seed", "generation"])
    rho = []
    for key, b in batches:
        if len(b) >= 4 and b["rlm_score"].nunique() > 1 and b["fit0"].nunique() > 1:
            rho.append({"method_name": key[0], "rho": stats.spearmanr(b["rlm_score"], b["fit0"]).statistic})
    rho = pd.DataFrame(rho)
    if len(rho):
        display(rho.groupby("method_name")["rho"].describe()[["count", "mean", "50%", "std"]]
                .rename(columns={"50%": "median"}).style.format(precision=3))
        fig, ax = plt.subplots(figsize=(8, 4))
        for m, g in rho.groupby("method_name"):
            ax.hist(g["rho"], bins=np.linspace(-1, 1, 21), histtype="step", color=COLOR[m],
                    linestyle=STYLE[m], linewidth=2, label=m)
        ax.axvline(0, color="#52514e", linewidth=1)
        ax.set(xlabel="Within-batch Spearman (prediction vs real)", ylabel="Batches",
               title="How well the RLM ranks each generation's offspring")
        ax.legend(frameon=False)
        fig.tight_layout()
        save(fig, "rlm_within_batch")

    auroc = []
    for m, d in lo.groupby("method_name"):
        v, inv = d.loc[d["valid"], "rlm_score"], d.loc[~d["valid"], "rlm_score"]
        if len(v) and len(inv):
            u = stats.mannwhitneyu(v, inv, alternative="two-sided").statistic
            auroc.append({"method": m, "valid": len(v), "invalid": len(inv), "AUROC": u / (len(v) * len(inv))})
    if auroc:
        display(pd.DataFrame(auroc).set_index("method").style.format({"AUROC": "{:.3f}"}))
    else:
        print("No invalid candidates in the log-only runs, so no AUROC.")

**Counterfactual screening value.** For each log-only generation batch: had
`topk_filter` kept only the predicted-best k% of this batch, how much better
would the kept candidates have been than the whole batch? It's shown as a
fraction of what a perfect ranker would gain: 1 = oracle, 0 = no better than
random, < 0 = worse than random. *Precision@k* is the share of the truly best
k% that the RLM would have kept.

In [ ]:
def screening_value(batch, frac):
    n = len(batch)
    k = max(1, int(round(n * frac)))
    if n < 2 or k >= n:
        return None
    by_pred = batch.sort_values("rlm_score", ascending=False)
    by_real = batch.sort_values("fit0", ascending=False)
    base = batch["fit0"].mean()
    pred_mean, oracle_mean = by_pred["fit0"].head(k).mean(), by_real["fit0"].head(k).mean()
    gain = oracle_mean - base
    return {
        "kept": f"top {frac:.0%}",
        "batch mean": base,
        "kept mean (RLM)": pred_mean,
        "kept mean (oracle)": oracle_mean,
        "lift captured": (pred_mean - base) / gain if gain > 1e-12 else np.nan,
        "precision@k": len(set(by_pred.index[:k]) & set(by_real.index[:k])) / k,
        "best kept (RLM)": by_pred["fit0"].head(k).max() >= batch["fit0"].max(),
    }


if not lo.empty:
    vals = []
    for (m, s, g), b in lo[lo["generation"] > 0].groupby(["method_name", "seed", "generation"]):
        for frac in (0.5, 0.25):
            r = screening_value(b, frac)
            if r:
                vals.append({"method": m, **r})
    vals = pd.DataFrame(vals)
    if len(vals):
        table = vals.groupby(["method", "kept"]).agg(
            batches=("batch mean", "size"),
            batch_mean=("batch mean", "mean"),
            kept_mean_rlm=("kept mean (RLM)", "mean"),
            kept_mean_oracle=("kept mean (oracle)", "mean"),
            lift_captured=("lift captured", "median"),
            precision_at_k=("precision@k", "mean"),
            best_kept=("best kept (RLM)", "mean"),
        )
        display(table.style.format(precision=3).format({"best_kept": "{:.0%}"}))
        print("lift_captured is the median over batches; best_kept is the share of batches "
              "where the RLM's kept set contains the batch's best candidate.")

**Calibration.** Mean real fitness per decile of predicted score. A rising line
means higher predictions do pick better algorithms on average, whatever the
absolute scale of the scores.

In [ ]:
if not lo.empty and lo["rlm_score"].nunique() > 10:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for m, d in lo.groupby("method_name"):
        d = d.copy()
        d["bin"] = pd.qcut(d["rlm_score"], 10, labels=False, duplicates="drop")
        g = d.groupby("bin").agg(pred=("rlm_score", "mean"), real=("fit0", "mean"), sem=("fit0", "sem"))
        ax.errorbar(g["pred"], g["real"], yerr=g["sem"], color=COLOR[m], linestyle=STYLE[m],
                    marker="o", markersize=7, linewidth=2, capsize=3, label=m)
    ax.set(xlabel="Mean predicted score (decile)", ylabel="Mean real AOCC (invalid = 0)",
           title="RLM calibration")
    ax.legend(frameon=False)
    fig.tight_layout()
    save(fig, "rlm_calibration")

## 6. Screened arms: what got evaluated

Quality of the offspring each arm actually spent real evaluations on
(generation > 0, so the shared random initial population is left out). If
screening works, the filtering arms should evaluate better offspring and fewer
crashing ones than the baseline. Their prediction-vs-real correlation is shown
for completeness, but it is **biased**: these arms only evaluate candidates
the RLM already ranked high, which narrows the range of predictions.

In [ ]:
off = cand[cand["generation"] > 0]
quality = off.groupby(["method_name", "seed"]).agg(
    mean_offspring=("fit0", "mean"), median_offspring=("fit0", "median"), invalid=("valid", lambda v: 1 - v.mean())
).reset_index()
q = quality.groupby("method_name")[["mean_offspring", "median_offspring", "invalid"]].mean().reindex(METHODS)

bias = []
for m in SCREENED:
    d = off[(off["method_name"] == m) & off["valid"] & off["rlm_score"].notna()]
    if len(d) >= 3 and d["rlm_score"].nunique() > 1:
        bias.append({"method_name": m, "Spearman (evaluated only)": stats.spearmanr(d["rlm_score"], d["fitness"]).statistic})
q = q.join(pd.DataFrame(bias).set_index("method_name") if bias else pd.DataFrame())
display(q.style.format({"mean_offspring": "{:.3f}", "median_offspring": "{:.3f}", "invalid": "{:.1%}"},
                       precision=3, na_rep="–"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=quality, x="method_name", y="mean_offspring", order=METHODS, palette=COLOR, width=0.55,
            fliersize=0, linewidth=1, ax=axes[0], hue="method_name", legend=False)
sns.stripplot(data=quality, x="method_name", y="mean_offspring", order=METHODS, color="#0b0b0b", size=5,
              jitter=0.12, ax=axes[0])
axes[0].set(xlabel="", ylabel="Mean offspring AOCC per run (invalid = 0)", title="Offspring quality")
sns.boxplot(data=quality, x="method_name", y="invalid", order=METHODS, palette=COLOR, width=0.55,
            fliersize=0, linewidth=1, ax=axes[1], hue="method_name", legend=False)
sns.stripplot(data=quality, x="method_name", y="invalid", order=METHODS, color="#0b0b0b", size=5,
              jitter=0.12, ax=axes[1])
axes[1].set(xlabel="", ylabel="Share of invalid offspring per run", title="Crashing candidates evaluated")
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
for ax in axes:
    hatch_boxes(ax, METHODS)
    ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
save(fig, "offspring_quality")

Mean offspring fitness per generation. `threshold_reject` runs about twice as
many generations as the others (it evaluates roughly half of each batch), so
its x-axis runs further.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for m in METHODS:
    g = off[off["method_name"] == m].groupby(["seed", "generation"])["fit0"].mean().groupby("generation")
    stats_g = g.agg(["mean", "sem"])
    ax.plot(stats_g.index, stats_g["mean"], color=COLOR[m], linestyle=STYLE[m], linewidth=2,
            marker="o", markersize=4, label=m)
    ax.fill_between(stats_g.index, stats_g["mean"] - stats_g["sem"], stats_g["mean"] + stats_g["sem"],
                    color=COLOR[m], alpha=0.12, linewidth=0)
ax.set(xlabel="Generation", ylabel="Mean offspring AOCC (± s.e.m. over runs)", title="Offspring quality per generation")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
save(fig, "offspring_per_generation")

## Best algorithm per method

The best algorithm each method found (over all its finished runs). The full
code is in the `solution` column of `logger.get_data()`.

In [ ]:
exp["best_fitness"] = exp["solution"].apply(lambda s: s.get("fitness"))
best = exp.loc[exp.groupby("method_name")["best_fitness"].idxmax()]
best = best.assign(algorithm=best["solution"].apply(lambda s: s.get("name")),
                   description=best["solution"].apply(lambda s: s.get("description")))
best.set_index("method_name").reindex(METHODS)[["seed", "best_fitness", "algorithm", "description"]]